# GP03 — CSV to GIS Dataset

CSV → DataFrame → GeoDataFrame → QA → GeoJSON / GeoPackage

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
df = pd.read_csv("data/gp03/city_assets.csv")
print(df.shape)
print(df.dtypes)
print(df.isna().sum())

In [ ]:
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["longitude"], df["latitude"]), crs="EPSG:4326")
print(gdf.crs)
print(gdf.geometry.geom_type.value_counts())
gdf.plot(figsize=(7, 6), color="navy")

In [ ]:
assert len(gdf) == len(df)
assert len(gdf) > 0
assert gdf.crs.to_string() == "EPSG:4326"
assert gdf.geometry.geom_type.eq("Point").all()
assert gdf.geometry.notna().all()
assert gdf.drop(columns="geometry").isna().sum().sum() == 0
assert gdf.geometry.is_valid.all()
assert gdf["asset_id"].is_unique
assert gdf.total_bounds.shape == (4,)
print("Rows:", len(gdf), "Bounds:", gdf.total_bounds)

In [ ]:
output_dir = Path("outputs/gp03")
output_dir.mkdir(parents=True, exist_ok=True)
geojson_path = output_dir / "city_assets.geojson"
gpkg_path = output_dir / "city_assets.gpkg"
gdf.to_file(geojson_path, driver="GeoJSON")
gdf.to_file(gpkg_path, layer="assets", driver="GPKG")
check = gpd.read_file(gpkg_path, layer="assets")
assert len(check) == len(gdf)
assert check.crs is not None
print("Export OK")

## Mini Challenge — Ενδεικτική λύση

In [ ]:
good = gdf[gdf["condition"] == "good"].copy()
assert len(good) > 0
assert (good["condition"] == "good").all()
good_path = output_dir / "good_assets.geojson"
good.to_file(good_path, driver="GeoJSON")
print(good_path)

## Τελικός έλεγχος

Το CSV μετατράπηκε σε επαληθευμένο GIS dataset με geometry, CRS και export.